# Team evidence for F1–F4

This notebook audits the A/B handoff measurements and Zuoheng’s separate E3 and E4 reruns after the B+ range fix. Each experiment is tied to a recorded Docker session and code version. The handoff uses Fenghao's E1/E5 and Cimy's E2–E4. The E1 session predates the sorted-list point-lookup fix, which did not change insertion. E5 uses the session after that fix. E2–E4 precede the B+ range fix. The post-fix label refers to the B+ correction. Compare indexes within each session. Comparing medians across the A/B handoff and a rerun on a different machine also includes hardware and load differences.

The unit is total elapsed time for a build and elapsed time per query for searches. Interquartile ranges show repeat variation within a configuration. The two reference implementations provide context and are excluded from feature rankings.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'results' / 'timings.csv').exists())
raw = pd.read_csv(root / 'results' / 'timings.csv',
                  dtype={'session_id': str, 'code_version': str})

# Explicit (session_id, code_version) pairs from the archived measurements.
HANDOFF = {
    'E1_build_memory': ('63191d74d176', '03d990e'),
    'E2_point_track_id': ('721d1d950c87', '0ff4b8e'),
    'E3_range_tempo': ('3d1109855014', '5a59833'),
    'E4_prefix_track_name': ('27c2b42c5b4f', '751ca9a'),
    'E5_insert_order': ('558bce9a341a', '870c3a9'),
}
# Zuoheng reran E3 and E4 with all seven indexes on zuoheng-mbp after
# the B+ range fix. Other experiments use the handoff measurements.
POST_FIX = {
    'E3_range_tempo': ('4bfebe4a211e', '0d49e62'),
    'E4_prefix_track_name': ('e32f04a9efaa', '0d49e62'),
}

indexes = ('unsorted_list', 'sorted_list', 'hash_table', 'avl_tree',
           'bplus_tree', 'ref_dict', 'ref_bisect')
configurations = dict(zip(HANDOFF, (6, 12, 6, 5, 24)))  # Per index.
keys = ['experiment', 'key_column', 'insert_order', 'n_keys',
        'operation', 'operation_param', 'index_type', 'index_param']
selected = []
for phase, sources in [('Handoff', HANDOFF), ('Post-fix', POST_FIX)]:
    for experiment, (session, version) in sources.items():
        rows = raw.loc[raw['experiment'].eq(experiment) & raw['session_id'].eq(session)].copy()
        counts = rows.groupby(keys, dropna=False)['rep'].agg(['count', 'nunique'])
        expected = configurations[experiment] * len(indexes)
        if (rows['code_version'].unique().tolist() != [version]
                or version.endswith('-dirty') or rows['machine_id'].nunique() != 1
                or not rows['in_docker'].eq(True).all()
                or set(rows['index_type']) != set(indexes)
                or len(counts) != expected or (counts['count'] < 5).any()
                or (counts['count'] != counts['nunique']).any()):
            raise ValueError(f'Invalid or incomplete selection: {experiment}, {session}, {version}')
        rows['phase'] = phase
        selected.append(rows)

chosen = pd.concat(selected, ignore_index=True)
chosen['time_ns'] = chosen['total_ns'].where(
    chosen['operation'].eq('build'), chosen['total_ns'] / chosen['n_ops'])
chosen['comparisons_per_op'] = chosen['comparisons'] / chosen['n_ops']
provenance = chosen.groupby(['phase', 'experiment', 'session_id', 'code_version',
                            'machine_id'], sort=False).size().rename('rows').reset_index()
display(provenance)

## 1. Summaries and growth estimates

Fit a line to `log(median time)` versus `log(n)` over the sampled sizes. The fitted slope is an effective exponent over this interval. An exponent near one agrees with a linear build or scan. A small positive exponent can describe logarithmic lookup work or size-dependent machine effects across this interval.

In [ ]:
summary = (chosen.groupby(['phase', *keys], dropna=False, sort=False)
           .agg(reps=('rep', 'size'), median_ns=('time_ns', 'median'),
                q1_ns=('time_ns', lambda x: x.quantile(.25)),
                q3_ns=('time_ns', lambda x: x.quantile(.75)),
                comparisons_per_op=('comparisons_per_op', 'median'),
                height=('height', 'median'), memory_bytes=('memory_bytes', 'median'))
           .reset_index())

scale = summary.loc[summary['experiment'].isin(['E1_build_memory', 'E2_point_track_id'])]
slopes = []
for (phase, experiment, operation, index), rows in scale.groupby(
        ['phase', 'experiment', 'operation', 'index_type']):
    exponent = np.polyfit(np.log(rows['n_keys']), np.log(rows['median_ns']), 1)[0]
    slopes.append({'phase': phase, 'experiment': experiment, 'operation': operation,
                   'index_type': index, 'log_log_slope': exponent})
display(pd.DataFrame(slopes).pivot(index=['phase', 'index_type'], columns='operation',
                                   values='log_log_slope').round(3))

largest = summary.loc[summary['experiment'].eq('E1_build_memory') & summary['n_keys'].eq(114000)].copy()
largest['build_ms'] = largest['median_ns'] / 1e6
largest['memory_MiB'] = largest['memory_bytes'] / 2**20
display(largest[['phase', 'index_type', 'build_ms', 'memory_MiB']].round(3))

## 2. Feature rankings and observed crossovers

Rank the five candidate indexes by median at each measured configuration. We report a crossover when the best candidate changes between adjacent sampled settings. Each reported interval gives the two settings that bracket the change. The top two may have overlapping repeat intervals, so check the IQRs before interpreting a narrow gap. E5 input order is categorical. The full-size table compares order sensitivity, without interpolating between orders.

In [ ]:
from IPython.display import Markdown

# (experiment, operation, x, scale of the reported median, unit)
rank_views = [
    ('E1_build_memory', 'build', 'n_keys', 1e6, 'ms'),
    ('E2_point_track_id', 'point_hit', 'n_keys', 1e3, 'µs'),
    ('E2_point_track_id', 'point_miss', 'n_keys', 1e3, 'µs'),
    ('E3_range_tempo', 'range', 'operation_param', 1e3, 'µs'),
    ('E4_prefix_track_name', 'prefix', 'operation_param', 1e3, 'µs'),
    ('E5_insert_order', 'build', 'insert_order', 1e6, 'ms'),
]

def subset(frame, experiment, operation):
    view = frame.loc[frame['experiment'].eq(experiment) & frame['operation'].eq(operation)]
    if experiment == 'E5_insert_order':
        view = view.loc[view['n_keys'].eq(100000)]
    return view


def interval(row, scale):
    return f"{row['median_ns']/scale:.3g} [{row['q1_ns']/scale:.3g}, {row['q3_ns']/scale:.3g}]"


candidates = summary.loc[~summary['index_type'].str.startswith('ref_')]
leader_records = []
changes = []
for experiment, operation, variable, scale, unit in rank_views:
    for phase, view in subset(candidates, experiment, operation).groupby('phase', sort=False):
        previous = None
        for value, config in view.groupby(variable, sort=True):
            leader, runner_up = config.sort_values(['median_ns', 'index_type']).iloc[:2].to_dict('records')
            name = leader['index_type']
            leader_records.append(dict(phase=phase, experiment=experiment, operation=operation,
                                       setting=value, unit=unit, winner=name,
                                       winner_median_IQR=interval(leader, scale),
                                       runner_up=runner_up['index_type'],
                                       runner_up_median_IQR=interval(runner_up, scale)))
            if variable != 'insert_order' and previous and name != previous[1]:
                changes.append(dict(phase=phase, experiment=experiment, operation=operation,
                                    interval=f'{previous[0]:g} to {value:g}',
                                    prior_winner=previous[1], next_winner=name))
            previous = (value, name)

leaders = pd.DataFrame(leader_records)
with pd.option_context('display.max_rows', None):
    display(leaders)
if changes:
    display(pd.DataFrame(changes))
else:
    display(Markdown('No best-candidate crossover between sampled sizes, selectivities, or prefix lengths.'))

# E5: the ratio quantifies order sensitivity even if the winning candidate is unchanged.
e5 = subset(summary, 'E5_insert_order', 'build')
order_ms = e5.pivot(index=['phase', 'index_type'], columns='insert_order', values='median_ns') / 1e6
orders = ['sorted', 'nearly_sorted', 'random', 'reversed']
order_ms = order_ms[orders]
order_ms['slowest_to_fastest'] = order_ms.max(axis=1) / order_ms.min(axis=1)
display(order_ms.round(3))

## 3. What the C-backed references tell us

`ref_dict` and `hash_table` both use hashing for point lookup. `ref_bisect` and the repaired `sorted_list` use binary search for point lookup. Their timings also reflect collision handling, memory layout, and the cost of collecting counters in the Python indexes. Range and prefix timings include traversing and materializing matches. The bisect reference finds both bounds and slices the relevant values, while the sorted-list index checks the upper bound in its Python loop. Treat these ratios as observations about the implementations rather than estimates of language overhead in isolation.

In [ ]:
comparisons = [
    ('E2_point_track_id', 'point_hit', 1.0, 'hash_table', 'ref_dict'),
    ('E2_point_track_id', 'point_miss', 0.0, 'hash_table', 'ref_dict'),
    ('E2_point_track_id', 'point_hit', 1.0, 'sorted_list', 'ref_bisect'),
    ('E2_point_track_id', 'point_miss', 0.0, 'sorted_list', 'ref_bisect'),
    ('E3_range_tempo', 'range', .01, 'sorted_list', 'ref_bisect'),
    ('E4_prefix_track_name', 'prefix', 3, 'sorted_list', 'ref_bisect'),
]
reference_rows = []
for experiment, operation, param, candidate, reference in comparisons:
    matching = subset(summary, experiment, operation)
    for phase, rows in matching.loc[matching['operation_param'].eq(param)].groupby('phase', sort=False):
        at_max_size = rows.loc[rows['n_keys'].eq(rows['n_keys'].max())].set_index('index_type')
        candidate_ns = at_max_size.loc[candidate, 'median_ns']
        reference_ns = at_max_size.loc[reference, 'median_ns']
        reference_rows.append(dict(phase=phase, operation=operation, parameter=param,
                                   candidate=candidate, reference=reference,
                                   candidate_µs=candidate_ns/1e3, reference_µs=reference_ns/1e3,
                                   candidate_to_reference=candidate_ns/reference_ns))
display(pd.DataFrame(reference_rows).round(3))

## 4. Figures

Points are medians and vertical bars span the IQR. Dashed lines identify references. Run this cell to regenerate the three figure panels for each explicitly selected phase.

In [ ]:
from matplotlib.lines import Line2D

output = root / 'analysis' / 'figures_team'
output.mkdir(exist_ok=True)
colors = dict(zip(indexes, ['#626A78', '#126782', '#9A4C10', '#8959A8',
                            '#237248', '#626A78', '#126782']))
labels = dict(zip(indexes, ['Unsorted list', 'Sorted list', 'Hash table', 'AVL tree',
                            'B+ tree', 'Dict ref.', 'Bisect ref.']))

def series_style(index):
    reference = index.startswith('ref_')
    return dict(color=colors[index], linestyle='--' if reference else '-',
                marker='s' if reference else 'o', markersize=3.5, linewidth=1.35)


def plot_view(ax, frame, experiment, operation, variable, scale, unit, title, xlabel, log_x):
    view = subset(frame, experiment, operation)
    if view.empty:
        ax.set_visible(False)
        return
    for index in indexes:
        line = view.loc[view['index_type'].eq(index)].sort_values(variable)
        if variable == 'insert_order':
            line = line.set_index(variable).loc[orders].reset_index()
            x = range(len(orders))
            ax.set_xticks(x, ['Sorted', 'Nearly sorted', 'Random', 'Reversed'])
        else:
            x = line[variable]
        median = line['median_ns'] / scale
        iqr = [(line['median_ns'] - line['q1_ns']) / scale,
               (line['q3_ns'] - line['median_ns']) / scale]
        ax.errorbar(x, median, yerr=iqr, capsize=2, elinewidth=.8, **series_style(index))
    if log_x:
        ax.set_xscale('log')
    if operation == 'prefix':
        ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_yscale('log')
    ax.set(xlabel=xlabel, ylabel=f'{"Total build" if operation == "build" else "Time/query"} ({unit})', title=title)
    ax.grid(axis='y', alpha=.2)
    ax.spines[['top', 'right']].set_visible(False)


panels = [
    ('F1_point', [
        (*rank_views[1], 'E2 · Point hits', 'Rows inserted', True),
        (*rank_views[2], 'E2 · Point misses', 'Rows inserted', True)]),
    ('F2_F3_queries', [
        (*rank_views[4], 'E4 · Prefix queries', 'Prefix length (characters)', False),
        (*rank_views[3], 'E3 · Tempo range queries', 'Selectivity (fraction of distinct keys)', True)]),
    ('F4_build', [
        (*rank_views[0], 'E1 · Catalog build', 'Rows inserted', True),
        (*rank_views[5], 'E5 · Build with 100,000 keys', 'Input order', False)]),
]
for phase, frame in summary.groupby('phase', sort=False):
    for name, views in panels:
        if not any(not subset(frame, view[0], view[1]).empty for view in views):
            continue
        fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.5), layout='constrained')
        for ax, view in zip(axes, views):
            plot_view(ax, frame, *view)
        handles = [Line2D([0], [0], label=labels[index], **series_style(index)) for index in indexes]
        fig.legend(handles=handles, loc='outside lower center', ncols=7, fontsize=8, frameon=False)
        if phase == 'Post-fix':
            title = 'After B+ range fix · zuoheng-mbp · 0d49e62'
        else:
            title = 'Handoff sessions · Median and interquartile range'
        fig.suptitle(title, fontsize=12)
        fig.savefig(output / f'{phase.lower()}_{name}.png', dpi=180, facecolor='white')
        plt.show()